# Pre-blend tile-to-tile disagreement, and the far-field perturbation test

Two questions the stitched output cannot answer about itself.

**1. How much do two tiles actually disagree before they are reconciled?**
After the blend they agree by construction, so any post-blend number is identically
zero and says nothing. The disagreement has to be captured *before* the operator runs,
which is what `--tiling.preblend_mode summary` writes.

This is the number that empirically checks the overlap width. At Δx ≈ 1.56 km and
Δt = 1 h, one model step advects 2.3 cells at 1 m/s and 6.9 at 3 m/s, so 8 cells past
each seam is a physically sound floor. If the disagreement has not flattened out by
the edge of the overlap, the width is too narrow.

**2. How much of that disagreement is the architecture rather than the physics?**
The U-Net's receptive field is wider than a tile and `GroupNorm` computes statistics
over the whole tile, so every cell is coupled to every other. The far-field
perturbation test separates the two: perturb a box far from the seam and watch the
induced response. Receptive-field coupling **decays with distance**; GroupNorm
coupling appears as a **distance-independent floor**. The size of that floor is the
structural limit on how well any two tiles can ever agree — and it is the number to
have in hand before anyone pays for a normalization retrain.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

plt.rcParams['figure.dpi'] = 110

## Part 1 — pre-blend disagreement

`preblend.zarr` holds `disagreement[time, seam, channel, offset]`: the RMS of
`|delta_A - delta_B|` reduced along each seam, in **normalized** (z-scored) units, so
channels are directly comparable.

In [ ]:
# ============== LOAD PRE-BLEND DISAGREEMENT ==============
# Written when the eval job runs with PREBLEND_MODE=summary (or full).
preblend_configs = [
    {
        'name': 'quintic blend',
        'key': 'quintic',
        'path': '/orcd/data/abodner/002/cody/inference_patch/'
                '2026-08-09-eval:Samudra_LLC:4-tile-blended-rollout/preblend.zarr',
        'desc': 'disagreement is a property of the model, not the window, but the '
                'blend changes the state each step so the runs do diverge',
    },
    # {
    #     'name': 'hard crop (no blend)',
    #     'key': 'crop',
    #     'path': '.../preblend.zarr',
    #     'desc': 'unreconciled state feeds forward; expect this to grow faster',
    # },
]

preblend = {}
for cfg in preblend_configs:
    preblend[cfg['key']] = xr.open_zarr(cfg['path'], consolidated=True)
    print(f"Loaded {cfg['name']}: {cfg['desc']}")
    print(f"   {dict(preblend[cfg['key']].sizes)}")

reference = preblend[preblend_configs[0]['key']]
seam_names = [str(s) for s in reference['seam'].values]
channel_names = [str(c) for c in reference['channel'].values]
n_offset = reference.sizes['offset']
print(f'\nseams:    {seam_names}')
print(f'channels: {len(channel_names)}  (e.g. {channel_names[:4]} ...)')
print(f'overlap:  {n_offset} cells')

In [ ]:
# ============== CHANNEL GROUPS ==============
# Group the flat channel list back into variables so the plots stay readable.
def channel_group(name):
    base, _, level = name.rpartition('_')
    return base if base else name

groups = {}
for index, name in enumerate(channel_names):
    groups.setdefault(channel_group(name), []).append(index)

print({k: f'{len(v)} levels' for k, v in groups.items()})

SURFACE_CHANNELS = {
    group: indices[0] for group, indices in groups.items() if indices
}
print('surface channel index per group:', SURFACE_CHANNELS)

### Disagreement across the overlap

Offset 0 is one edge of the 16-cell overlap and offset 15 the other. A curve that is
still high at the edges means the tiles disagree right where one of them is handing
over to the other — the case the overlap width is supposed to prevent.

In [ ]:
# ============== DISAGREEMENT vs OFFSET ACROSS THE OVERLAP ==============
CURVE_STEPS = None    # None -> first, middle, last
cfg = preblend_configs[0]
data = preblend[cfg['key']]
n_steps = data.sizes['time']
if CURVE_STEPS is None:
    CURVE_STEPS = sorted({0, n_steps // 2, n_steps - 1})

offsets = np.arange(n_offset)
fig, axes = plt.subplots(
    1, len(seam_names), figsize=(3.6 * len(seam_names), 3.2),
    squeeze=False, sharey=True,
)
for col, seam in enumerate(seam_names):
    ax = axes[0][col]
    for step in CURVE_STEPS:
        values = data['disagreement'].isel(time=step, seam=col).values
        ax.plot(offsets, values.mean(axis=0), lw=1.6, label=f'step {step}')
    ax.set_title(seam, fontsize=9)
    ax.set_xlabel('offset across overlap (cells)')
    if col == 0:
        ax.set_ylabel('RMS |delta_A - delta_B|\n(normalized units)')
axes[0][0].legend(fontsize=7)
fig.suptitle(f"{cfg['name']}: pre-blend disagreement across each seam, "
             'averaged over all channels')
fig.tight_layout()
plt.show()

### Disagreement vs autoregressive step

The question that decides whether blend-at-inference is enough: does disagreement
saturate, or does it compound? A rising curve means each step hands the next a state
the two tiles read differently, and no amount of blending fixes the cause.

In [ ]:
# ============== DISAGREEMENT vs AUTOREGRESSIVE STEP ==============
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))

# Left: per seam, averaged over channels and the overlap.
for cfg in preblend_configs:
    data = preblend[cfg['key']]
    for col, seam in enumerate(seam_names):
        series = data['disagreement'].isel(seam=col).mean(dim=['channel', 'offset'])
        axes[0].plot(series.values, lw=1.4, label=f"{cfg['name']} — {seam}")
axes[0].set_xlabel('autoregressive step')
axes[0].set_ylabel('mean RMS disagreement')
axes[0].set_title('per seam')
axes[0].legend(fontsize=7)

# Right: per variable group, averaged over seams and the overlap.
cfg = preblend_configs[0]
data = preblend[cfg['key']]
for group, indices in groups.items():
    series = (
        data['disagreement'].isel(channel=indices)
        .mean(dim=['seam', 'channel', 'offset'])
    )
    axes[1].plot(series.values, lw=1.4, label=group)
axes[1].set_xlabel('autoregressive step')
axes[1].set_title(f"{cfg['name']}: per variable")
axes[1].legend(fontsize=7, ncol=2)

fig.suptitle('Pre-blend tile-to-tile disagreement over the rollout')
fig.tight_layout()
plt.show()

In [ ]:
# ============== DISAGREEMENT BY DEPTH ==============
# Deep levels are quieter, so a disagreement that is flat with depth points at the
# architecture rather than at the physics.
DEPTH_GROUPS = [g for g in ('U', 'V', 'Theta', 'Salt') if g in groups]
STEP_FOR_DEPTH = -1

cfg = preblend_configs[0]
data = preblend[cfg['key']]
fig, ax = plt.subplots(figsize=(5.5, 4.0))
for group in DEPTH_GROUPS:
    indices = groups[group]
    profile = (
        data['disagreement'].isel(time=STEP_FOR_DEPTH, channel=indices)
        .mean(dim=['seam', 'offset']).values
    )
    ax.plot(profile, np.arange(len(profile)), lw=1.5, label=group)
ax.invert_yaxis()
ax.set_xlabel('mean RMS disagreement (normalized units)')
ax.set_ylabel('depth level index')
ax.set_title(f"{cfg['name']}: disagreement vs depth at step {STEP_FOR_DEPTH}")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## Part 2 — far-field perturbation test

Run with `PERTURBATION=true` (its own job, its own `perturbation.zarr`). Each step the
model is re-run from a state perturbed inside a box far from the seam, starting from
the *same* control state rather than as a second diverging rollout — so this measures
one-step coupling, not accumulated drift.

`response_by_distance[time, tile, channel, distance]` is the RMS response binned by
distance from the perturbed box.

**Read it like this.** A response that decays to the noise floor is receptive-field
coupling and is bounded by the overlap. A response that flattens out at some non-zero
level is `GroupNorm` coupling every cell to the whole tile through shared spatial
statistics — that floor is a hard limit on tile-to-tile agreement that no overlap
width can remove, and consistency training can only reduce.

In [ ]:
# ============== LOAD PERTURBATION RESPONSE ==============
PERTURBATION_PATH = (
    '/orcd/data/abodner/002/cody/inference_patch/'
    '2026-08-09-eval:Samudra_LLC:4-tile-perturbation/perturbation.zarr'
)
perturb = xr.open_zarr(PERTURBATION_PATH, consolidated=True)
print(dict(perturb.sizes))
for key in ('perturbation_centre', 'perturbation_box', 'perturbation_amplitude',
            'response_map_channel', 'note'):
    if key in perturb.attrs:
        print(f'{key:24s} {perturb.attrs[key]}')

In [ ]:
# ============== RESPONSE vs DISTANCE FROM THE PERTURBED BOX ==============
distance = perturb['distance'].values
PERTURB_STEPS = sorted({0, perturb.sizes['time'] // 2, perturb.sizes['time'] - 1})

fig, axes = plt.subplots(1, 2, figsize=(11, 4.0))

# Left: log-scale response, all tiles, several steps.
for step in PERTURB_STEPS:
    curve = perturb['response_by_distance'].isel(time=step).mean(
        dim=['tile', 'channel']
    ).values
    axes[0].semilogy(distance, curve, lw=1.6, label=f'step {step}')
axes[0].set_xlabel('distance from perturbed box (cells)')
axes[0].set_ylabel('RMS response (normalized units)')
axes[0].set_title('response decay — flat tail means GroupNorm coupling')
axes[0].legend(fontsize=8)

# Right: normalized by the near-field value, which makes the shape comparable
# across steps and is where a floor becomes obvious.
for step in PERTURB_STEPS:
    curve = perturb['response_by_distance'].isel(time=step).mean(
        dim=['tile', 'channel']
    ).values
    axes[1].semilogy(distance, curve / curve[0], lw=1.6, label=f'step {step}')
axes[1].set_xlabel('distance from perturbed box (cells)')
axes[1].set_ylabel('response / near-field response')
axes[1].set_title('normalized shape')
axes[1].legend(fontsize=8)

fig.suptitle('Far-field perturbation: does the response decay, or hit a floor?')
fig.tight_layout()
plt.show()

In [ ]:
# ============== QUANTIFY THE FLOOR ==============
# The headline number: how much of the near-field response is still present at the
# far edge of the tile. A few percent is receptive-field leakage; a large fraction is
# a normalization floor.
FAR_FRACTION = 0.75   # "far" = beyond this fraction of the max binned distance

far = distance >= FAR_FRACTION * distance.max()
rows = []
for step in range(perturb.sizes['time']):
    curve = perturb['response_by_distance'].isel(time=step).mean(
        dim=['tile', 'channel']
    ).values
    rows.append({
        'step': step,
        'near_field': float(curve[0]),
        'far_field': float(np.nanmean(curve[far])),
        'floor_fraction': float(np.nanmean(curve[far]) / curve[0]),
    })

floor = pd.DataFrame(rows)
pd.set_option('display.float_format', lambda v: f'{v: .4e}')
display(floor)
print(f"\nMean far-field floor as a fraction of near-field response: "
      f"{floor['floor_fraction'].mean():.3%}")

In [ ]:
# ============== RESPONSE MAP ==============
# The visual companion: if the response is a broad wash rather than a blob centred on
# the perturbation, the coupling is not local.
MAP_STEP = PERTURB_STEPS[-1]
n_tiles = perturb.sizes['tile']

fig, axes = plt.subplots(1, n_tiles, figsize=(3.4 * n_tiles, 3.6), squeeze=False)
maps = perturb['response_map'].isel(time=MAP_STEP).values
vmax = np.nanpercentile(maps, 99)
for tile_index in range(n_tiles):
    ax = axes[0][tile_index]
    im = ax.imshow(maps[tile_index], origin='lower', vmin=0, vmax=vmax, cmap='inferno')
    ax.set_title(f"tile {int(perturb['tile'].values[tile_index])}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axes[0], shrink=0.8,
             label=f"|response| ({perturb.attrs.get('response_map_channel', '')})")
fig.suptitle(f'One-step response to a far-field perturbation, step {MAP_STEP}\n'
             f"box {perturb.attrs.get('perturbation_box', '?')} cells at "
             f"{perturb.attrs.get('perturbation_centre', '?')}")
plt.show()

## What to conclude

* **Disagreement flat across the overlap and steady over the rollout** — the width is
  adequate and blend-at-inference is doing its job. Move on to grouped training.
* **Disagreement still large at the overlap edges** — widen the overlap before
  anything else; the operator cannot reconcile what it never sees.
* **Disagreement growing with step** — the tiles are diverging faster than the blend
  reconciles them. That is the case for `L_overlap` (step 5), which attacks the cause
  rather than the symptom.
* **A large perturbation floor** — the disagreement has an architectural component
  that no overlap width and no window removes. That is the evidence for switching to
  a per-pixel channel LayerNorm, which has no spatial coupling at all; a small floor
  means the retrain is not worth it and the overlap plus `L_overlap` suffices.
